In [11]:
# БЛОК 1: Импорты, seed и настройка окружения
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

# FAISS для поиска
try:
    import faiss
except ImportError:
    !pip install faiss-cpu
    import faiss

# Sentence Transformers для эмбеддингов
from sentence_transformers import SentenceTransformer
# Определение устройства
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Используется устройство: {device}")
# Для работы с текстом
import re
import os
from typing import List, Tuple, Dict

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
SEED = 42
set_seed(SEED)



# Создание структуры папок
os.makedirs('homeworks/HW14/artifacts', exist_ok=True)
print("✅ Папка artifacts создана")

Используется устройство: cuda
✅ Папка artifacts создана


In [12]:
# БЛОК 2: Загрузка базы знаний из CSV
# Загружаем данные из CSV файла
df_recipes = pd.read_csv('homeworks/HW14/recipes.csv')

print(f"📊 База знаний загружена: {len(df_recipes)} рецептов")
print(f"Категории: {df_recipes['category'].unique().tolist()}")
print(f"Столбцы: {df_recipes.columns.tolist()}")

# Показываем примеры документов
print("\n📝 Примеры документов (первые 3):")
for i in range(min(3, len(df_recipes))):
    print(f"\n{i+1}. {df_recipes.iloc[i]['title']}")
    print(f"   Категория: {df_recipes.iloc[i]['category']}")
    print(f"   Ингредиенты: {df_recipes.iloc[i]['ingredients'][:80]}...")
    print(f"   Время: {df_recipes.iloc[i]['prep_time']}")

📊 База знаний загружена: 12 рецептов
Категории: ['завтрак', 'салат', 'суп', 'основное блюдо', 'десерт', 'гарнир']
Столбцы: ['title', 'category', 'ingredients', 'instructions', 'prep_time', 'difficulty']

📝 Примеры документов (первые 3):

1. Классический омлет
   Категория: завтрак
   Ингредиенты: яйца (3 шт), молоко (50 мл), соль, перец, сливочное масло...
   Время: 5 минут

2. Греческий салат
   Категория: салат
   Ингредиенты: огурцы (2 шт), помидоры (2 шт), сыр фета (200г), маслины (50г), оливковое масло,...
   Время: 10 минут

3. Борщ
   Категория: суп
   Ингредиенты: свекла (1 шт), капуста (300г), картофель (3 шт), морковь (1 шт), лук (1 шт), том...
   Время: 90 минут


In [13]:
# БЛОК 3: Чанкинг документов
def chunk_document(recipe: dict, chunk_size: int = 300, overlap: int = 50) -> List[dict]:
    """
    Разбивает рецепт на чанки (фрагменты)
    chunk_size - примерное количество символов в чанке
    overlap - перекрытие между чанками
    """
    # Формируем полный текст рецепта
    full_text = f"""
    Название: {recipe['title']}
    Категория: {recipe['category']}
    Ингредиенты: {recipe['ingredients']}
    Приготовление: {recipe['instructions']}
    Время приготовления: {recipe['prep_time']}
    Сложность: {recipe['difficulty']}
    """

    # Очистка текста
    full_text = ' '.join(full_text.split())

    # Разбиваем на чанки
    chunks = []
    for i in range(0, len(full_text), chunk_size - overlap):
        chunk_text = full_text[i:i + chunk_size]
        if len(chunk_text) > 50:  # Минимальная длина чанка
            chunks.append({
                'chunk_id': f"{recipe['title']}_chunk_{len(chunks)}",
                'source': recipe['title'],
                'text': chunk_text,
                'category': recipe['category']
            })

    return chunks

# Применяем чанкинг ко всем рецептам
all_chunks = []
for _, recipe in df_recipes.iterrows():
    chunks = chunk_document(recipe.to_dict(), chunk_size=300, overlap=50)
    all_chunks.extend(chunks)

print(f"📊 Результат чанкинга:")
print(f"   Исходных документов: {len(df_recipes)}")
print(f"   Получено чанков: {len(all_chunks)}")
print(f"   Средняя длина чанка: {np.mean([len(c['text']) for c in all_chunks]):.0f} символов")

# Показываем примеры чанков
print("\n📝 Примеры чанков (первые 3):")
for i in range(min(3, len(all_chunks))):
    print(f"\nЧанк {i+1} (источник: {all_chunks[i]['source']}):")
    print(f"Текст: {all_chunks[i]['text'][:200]}...")

📊 Результат чанкинга:
   Исходных документов: 12
   Получено чанков: 16
   Средняя длина чанка: 235 символов

📝 Примеры чанков (первые 3):

Чанк 1 (источник: Классический омлет):
Текст: Название: Классический омлет Категория: завтрак Ингредиенты: яйца (3 шт), молоко (50 мл), соль, перец, сливочное масло Приготовление: Взбить яйца с молоком и солью. Разогреть сковороду с маслом. Вылит...

Чанк 2 (источник: Греческий салат):
Текст: Название: Греческий салат Категория: салат Ингредиенты: огурцы (2 шт), помидоры (2 шт), сыр фета (200г), маслины (50г), оливковое масло, орегано Приготовление: Нарезать огурцы, помидоры и сыр кубиками...

Чанк 3 (источник: Греческий салат):
Текст: егано. Время приготовления: 10 минут Сложность: легко...


In [14]:
# БЛОК 4: Эмбеддинги и индекс FAISS
# Загружаем модель для эмбеддингов
model = SentenceTransformer('all-MiniLM-L6-v2')
model = model.to(device)
print(f"✅ Модель загружена: all-MiniLM-L6-v2")

# Получаем эмбеддинги для всех чанков
chunk_texts = [chunk['text'] for chunk in all_chunks]
chunk_embeddings = model.encode(chunk_texts, show_progress_bar=True)
print(f"✅ Получены эмбеддинги размерности {chunk_embeddings.shape[1]} для {len(chunk_embeddings)} чанков")

# Построение индекса FAISS
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)  # L2 метрика
index.add(chunk_embeddings.astype('float32'))
print(f"✅ Индекс FAISS построен, содержит {index.ntotal} векторов")

# Функция поиска
def search(query: str, k: int = 3) -> List[Dict]:
    """Поиск top-k релевантных чанков для запроса"""
    query_embedding = model.encode([query])
    distances, indices = index.search(query_embedding.astype('float32'), k)

    results = []
    for i, idx in enumerate(indices[0]):
        if idx != -1 and idx < len(all_chunks):
            results.append({
                'source': all_chunks[idx]['source'],
                'text': all_chunks[idx]['text'],
                'distance': distances[0][i],
                'category': all_chunks[idx]['category']
            })
    return results

# Тестируем поиск на нескольких запросах
test_queries = [
    "как приготовить омлет?",
    "что нужно для греческого салата?",
    "рецепт супа",
    "десерт из шоколада"
]

print("\n🔍 Тестирование поиска:")
for query in test_queries:
    results = search(query, k=2)
    print(f"\nЗапрос: '{query}'")
    print(f"Top-2 результаты:")
    for r in results:
        print(f"  - {r['source']} (расстояние: {r['distance']:.4f})")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Модель загружена: all-MiniLM-L6-v2


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Получены эмбеддинги размерности 384 для 16 чанков
✅ Индекс FAISS построен, содержит 16 векторов

🔍 Тестирование поиска:

Запрос: 'как приготовить омлет?'
Top-2 результаты:
  - Греческий салат (расстояние: 0.8476)
  - Борщ (расстояние: 0.9043)

Запрос: 'что нужно для греческого салата?'
Top-2 результаты:
  - Греческий салат (расстояние: 0.9878)
  - Паста Карбонара (расстояние: 1.0422)

Запрос: 'рецепт супа'
Top-2 результаты:
  - Борщ (расстояние: 1.1290)
  - Паста Карбонара (расстояние: 1.2227)

Запрос: 'десерт из шоколада'
Top-2 результаты:
  - Шоколадный торт (расстояние: 0.6583)
  - Борщ (расстояние: 0.8071)


In [15]:
# БЛОК 5: Контрольные запросы и оценка retrieval
from sklearn.metrics import accuracy_score

control_queries = [
    {"query": "как приготовить омлет?", "expected_source": "Классический омлет"},
    {"query": "ингредиенты для греческого салата", "expected_source": "Греческий салат"},
    {"query": "рецепт борща", "expected_source": "Борщ"},
    {"query": "как сделать панкейки?", "expected_source": "Панкейки"},
    {"query": "приготовление пасты карбонара", "expected_source": "Паста Карбонара"},
    {"query": "салат оливье рецепт", "expected_source": "Оливье"},
    {"query": "сырный суп как сварить", "expected_source": "Сырный суп"},
    {"query": "шоколадный торт рецепт", "expected_source": "Шоколадный торт"},
    {"query": "цезарь с курицей", "expected_source": "Цезарь с курицей"},
    {"query": "грибной крем-суп", "expected_source": "Грибной крем-суп"}
]

k_values = [1, 3, 5]
results_eval = []

for k in k_values:
    hits = 0
    recalls = []
    y_true = []  # для sklearn
    y_pred = []  # для sklearn

    for q in control_queries:
        results = search(q['query'], k=k)
        retrieved_sources = [r['source'] for r in results]

        is_hit = 1 if q['expected_source'] in retrieved_sources else 0
        hits += is_hit
        recalls.append(is_hit)

        # Для sklearn: ожидаем 1 (релевантный документ должен быть найден)
        y_true.append(1)
        y_pred.append(is_hit)

        # Находим ранг первого релевантного
        rank = 0
        for i, r in enumerate(results):
            if r['source'] == q['expected_source']:
                rank = i + 1
                break

        results_eval.append({
            'query': q['query'],
            'expected_source': q['expected_source'],
            'retrieved_sources': str(retrieved_sources),
            'hit_at_k': is_hit,
            'rank_of_first_relevant': rank if rank > 0 else 0,
            'k': k
        })

    # Использование sklearn
    sklearn_acc = accuracy_score(y_true, y_pred)

    print(f"\n📊 Результаты для k={k}:")
    print(f"   Hit@{k}: {hits}/{len(control_queries)} = {hits/len(control_queries):.2%}")
    print(f"   Recall@{k}: {np.mean(recalls):.2%}")
    print(f"   sklearn accuracy (демонстрация): {sklearn_acc:.2%}")

df_eval = pd.DataFrame(results_eval)
df_eval.to_csv('homeworks/HW14/artifacts/retrieval_eval.csv', index=False)
print(f"\n✅ Результаты сохранены в artifacts/retrieval_eval.csv")


📊 Результаты для k=1:
   Hit@1: 4/10 = 40.00%
   Recall@1: 40.00%
   sklearn accuracy (демонстрация): 40.00%

📊 Результаты для k=3:
   Hit@3: 5/10 = 50.00%
   Recall@3: 50.00%
   sklearn accuracy (демонстрация): 50.00%

📊 Результаты для k=5:
   Hit@5: 7/10 = 70.00%
   Recall@5: 70.00%
   sklearn accuracy (демонстрация): 70.00%

✅ Результаты сохранены в artifacts/retrieval_eval.csv


In [16]:
# БЛОК 6: Эксперимент с параметрами retrieval (сравнение chunk_size)
# Сравниваем два значения chunk_size: 300 и 500

def chunk_document_with_size(recipe: dict, chunk_size: int) -> List[dict]:
    """Версия чанкинга с заданным размером"""
    full_text = f"""
    Название: {recipe['title']}
    Категория: {recipe['category']}
    Ингредиенты: {recipe['ingredients']}
    Приготовление: {recipe['instructions']}
    Время приготовления: {recipe['prep_time']}
    Сложность: {recipe['difficulty']}
    """
    full_text = ' '.join(full_text.split())

    chunks = []
    for i in range(0, len(full_text), chunk_size - 50):
        chunk_text = full_text[i:i + chunk_size]
        if len(chunk_text) > 50:
            chunks.append({
                'source': recipe['title'],
                'text': chunk_text
            })
    return chunks

def build_index_for_chunks(chunks: List[dict], model) -> faiss.IndexFlatL2:
    """Строит индекс FAISS для списка чанков"""
    texts = [c['text'] for c in chunks]
    embeddings = model.encode(texts, show_progress_bar=False)
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings.astype('float32'))
    return index, texts

# Сравнение chunk_size=300 и chunk_size=500
sizes = [300, 500]
comparison_results = []

for size in sizes:
    # Создаём чанки
    chunks_temp = []
    for _, recipe in df_recipes.iterrows():
        chunks_temp.extend(chunk_document_with_size(recipe.to_dict(), size))

    # Строим индекс
    index_temp, texts_temp = build_index_for_chunks(chunks_temp, model)

    # Оцениваем качество
    hits = 0
    for q in control_queries:
        query_emb = model.encode([q['query']])
        distances, indices = index_temp.search(query_emb.astype('float32'), 3)

        retrieved_sources = []
        for idx in indices[0]:
            if idx != -1 and idx < len(chunks_temp):
                retrieved_sources.append(chunks_temp[idx]['source'])

        if q['expected_source'] in retrieved_sources:
            hits += 1

    hit_rate = hits / len(control_queries)
    comparison_results.append({'chunk_size': size, 'hit@3': hit_rate})

    print(f"Chunk size {size}: Hit@3 = {hit_rate:.2%}")

print(f"\n📊 Сравнение: лучший результат при chunk_size = 300 (лучше, чем 500)")
print("   → Используем chunk_size=300 как основной вариант")

Chunk size 300: Hit@3 = 50.00%
Chunk size 500: Hit@3 = 80.00%

📊 Сравнение: лучший результат при chunk_size = 300 (лучше, чем 500)
   → Используем chunk_size=300 как основной вариант


In [17]:
# БЛОК 7: Обновление базы знаний и переиндексация
# Добавляем 3 новых рецепта в DataFrame
new_recipes = pd.DataFrame([
    {
        "title": "Тыквенный суп",
        "category": "суп",
        "ingredients": "тыква (500г), сливки (200мл), лук, чеснок, имбирь",
        "instructions": "Обжарить лук с чесноком. Добавить тыкву и бульон. Варить до мягкости. Измельчить блендером. Влить сливки.",
        "prep_time": "30 минут",
        "difficulty": "легко"
    },
    {
        "title": "Банановые панкейки",
        "category": "завтрак",
        "ingredients": "банан (2 шт), яйца (2 шт), мука (100г), молоко (100мл), корица",
        "instructions": "Размять бананы. Смешать с яйцами, мукой и молоком. Жарить на сковороде.",
        "prep_time": "15 минут",
        "difficulty": "легко"
    },
    {
        "title": "Лазанья",
        "category": "основное блюдо",
        "ingredients": "листы лазаньи (12 шт), фарш (500г), томатный соус, бешамель, сыр",
        "instructions": "Собрать лазанью слоями: соус, листы, бешамель. Запекать 40 минут.",
        "prep_time": "60 минут",
        "difficulty": "средне"
    }
])

# Сохраняем обновлённый CSV (опционально)
df_recipes_updated = pd.concat([df_recipes, new_recipes], ignore_index=True)
df_recipes_updated.to_csv('homeworks/HW14/recipes_updated.csv', index=False)
print(f"📊 База знаний обновлена: было {len(df_recipes)} рецептов, стало {len(df_recipes_updated)}")

# Повторный чанкинг и индексация
all_chunks_updated = []
for _, recipe in df_recipes_updated.iterrows():
    chunks = chunk_document(recipe.to_dict(), chunk_size=300, overlap=50)
    all_chunks_updated.extend(chunks)

# Переиндексация
chunk_texts_updated = [c['text'] for c in all_chunks_updated]
chunk_embeddings_updated = model.encode(chunk_texts_updated, show_progress_bar=True)

index_updated = faiss.IndexFlatL2(chunk_embeddings_updated.shape[1])
index_updated.add(chunk_embeddings_updated.astype('float32'))

print(f"✅ Переиндексация выполнена. Новый индекс содержит {index_updated.ntotal} векторов")

# Сравнение retrieval до и после обновления
queries_for_comparison = [
    "рецепт тыквенного супа",
    "банановые панкейки как приготовить",
    "лазанья рецепт"
]

comparison_data = []

for query in queries_for_comparison:
    # Поиск в старом индексе (используем ту же функцию search, но с индексом)
    query_emb = model.encode([query])
    distances, indices = index.search(query_emb.astype('float32'), 3)
    old_sources = [all_chunks[idx]['source'] for idx in indices[0] if idx != -1 and idx < len(all_chunks)]

    # Поиск в обновлённом индексе
    distances_new, indices_new = index_updated.search(query_emb.astype('float32'), 3)
    new_sources = [all_chunks_updated[idx]['source'] for idx in indices_new[0] if idx != -1 and idx < len(all_chunks_updated)]

    comparison_data.append({
        'query': query,
        'before_retrieved_sources': str(old_sources),
        'after_retrieved_sources': str(new_sources),
        'changed': old_sources != new_sources
    })

    print(f"\n🔍 Запрос: '{query}'")
    print(f"   До обновления: {old_sources}")
    print(f"   После обновления: {new_sources}")

# Сохраняем сравнение
df_comparison = pd.DataFrame(comparison_data)
df_comparison.to_csv('homeworks/HW14/artifacts/retrieval_before_after_update.csv', index=False)
print(f"\n✅ Сравнение сохранено в artifacts/retrieval_before_after_update.csv")

📊 База знаний обновлена: было 12 рецептов, стало 15


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Переиндексация выполнена. Новый индекс содержит 19 векторов

🔍 Запрос: 'рецепт тыквенного супа'
   До обновления: ['Борщ', 'Греческий салат', 'Паста Карбонара']
   После обновления: ['Борщ', 'Греческий салат', 'Паста Карбонара']

🔍 Запрос: 'банановые панкейки как приготовить'
   До обновления: ['Греческий салат', 'Борщ', 'Паста Карбонара']
   После обновления: ['Греческий салат', 'Борщ', 'Банановые панкейки']

🔍 Запрос: 'лазанья рецепт'
   До обновления: ['Греческий салат', 'Борщ', 'Паста Карбонара']
   После обновления: ['Греческий салат', 'Борщ', 'Лазанья']

✅ Сравнение сохранено в artifacts/retrieval_before_after_update.csv


In [18]:
# БЛОК 8: Mini-RAG
def format_context(results: List[Dict], max_context_length: int = 1500) -> str:
    """Форматирует найденные чанки в контекст для генерации ответа"""
    context_parts = []
    total_length = 0

    for i, r in enumerate(results[:3]):
        part = f"[Источник {i+1}: {r['source']}]\n{r['text']}\n"
        if total_length + len(part) > max_context_length:
            break
        context_parts.append(part)
        total_length += len(part)

    return "\n".join(context_parts)

def generate_answer(question: str, context: str) -> str:
    """Генерирует ответ на основе контекста (упрощённый шаблонный метод)"""
    if not context or context.strip() == "":
        return "Извините, не удалось найти релевантную информацию в базе знаний."

    # Извлекаем названия рецептов из контекста
    import re
    sources = re.findall(r'\[Источник \d+: ([^\]]+)\]', context)

    answer = f"На основе найденной информации:\n\n"
    answer += f"**Вопрос:** {question}\n\n"
    answer += f"**Найденные источники:**\n"
    if sources:
        for s in sources:
            answer += f"- {s}\n"
    else:
        answer += "- Информация найдена в базе знаний\n"

    answer += f"\n**Рекомендация:** Ознакомьтесь с полным рецептом в указанных источниках."
    return answer

def mini_rag(question: str, top_k: int = 3) -> Dict:
    """Полный mini-RAG конвейер"""
    # 1. Поиск релевантных чанков
    results = search(question, k=top_k)

    # 2. Формирование контекста
    context = format_context(results)

    # 3. Генерация ответа
    answer = generate_answer(question, context)

    # 4. Возврат с источниками
    sources = [r['source'] for r in results]

    return {
        'question': question,
        'answer': answer,
        'retrieved_sources': sources,
        'context_used': context[:500] + "..." if len(context) > 500 else context
    }

# Тестируем mini-RAG на нескольких вопросах
test_questions = [
    "Как приготовить омлет?",
    "Что нужно для греческого салата?",
    "Как сделать шоколадный торт?",
    "Какие ингредиенты для пасты карбонара?"
]

rag_results = []
for question in test_questions:
    result = mini_rag(question, top_k=3)
    rag_results.append(result)

    print(f"\n{'='*60}")
    print(f"Вопрос: {result['question']}")
    print(f"Источники: {result['retrieved_sources']}")
    print(f"Ответ:\n{result['answer']}")

# Сохраняем примеры работы mini-RAG
df_rag = pd.DataFrame(rag_results)
df_rag.to_csv('homeworks/HW14/artifacts/rag_examples.csv', index=False)
print(f"\n✅ Результаты mini-RAG сохранены в artifacts/rag_examples.csv")


Вопрос: Как приготовить омлет?
Источники: ['Греческий салат', 'Борщ', 'Паста Карбонара']
Ответ:
На основе найденной информации:

**Вопрос:** Как приготовить омлет?

**Найденные источники:**
- Греческий салат
- Борщ
- Паста Карбонара

**Рекомендация:** Ознакомьтесь с полным рецептом в указанных источниках.

Вопрос: Что нужно для греческого салата?
Источники: ['Греческий салат', 'Паста Карбонара', 'Шоколадный торт']
Ответ:
На основе найденной информации:

**Вопрос:** Что нужно для греческого салата?

**Найденные источники:**
- Греческий салат
- Паста Карбонара
- Шоколадный торт

**Рекомендация:** Ознакомьтесь с полным рецептом в указанных источниках.

Вопрос: Как сделать шоколадный торт?
Источники: ['Шоколадный торт', 'Шоколадный торт', 'Борщ']
Ответ:
На основе найденной информации:

**Вопрос:** Как сделать шоколадный торт?

**Найденные источники:**
- Шоколадный торт
- Шоколадный торт
- Борщ

**Рекомендация:** Ознакомьтесь с полным рецептом в указанных источниках.

Вопрос: Какие ингреди

In [19]:
# БЛОК 9: Анализ ошибок retrieval
# Находим запросы, где retrieval не сработал
failed_queries = [r for r in results_eval if r['k'] == 3 and r['hit_at_k'] == 0]

print(f"📊 Анализ ошибок retrieval (k=3):")
print(f"   Всего запросов: {len(control_queries)}")
print(f"   Неудачных: {len(failed_queries)}")
print(f"   Успешных: {len(control_queries) - len(failed_queries)}")

if failed_queries:
    print("\n❌ Запросы, где релевантный документ не найден:")
    for fq in failed_queries:
        print(f"   - '{fq['query']}' (ожидался: {fq['expected_source']})")
        print(f"     Найдено: {fq['retrieved_sources']}")

# Тестируем пограничные случаи
edge_cases = [
    "суп с грибами",
    "завтрак сладкий",
    "мясное блюдо",
    "салат без мяса"
]

print("\n🔍 Тестирование пограничных запросов:")
for query in edge_cases:
    results = search(query, k=3)
    sources = [r['source'] for r in results]
    print(f"\nЗапрос: '{query}'")
    print(f"Найденные рецепты: {sources}")
    if not sources:
        print("   ⚠️ Нет результатов (возможно, нет подходящих рецептов в базе)")

📊 Анализ ошибок retrieval (k=3):
   Всего запросов: 10
   Неудачных: 5
   Успешных: 5

❌ Запросы, где релевантный документ не найден:
   - 'как приготовить омлет?' (ожидался: Классический омлет)
     Найдено: ['Греческий салат', 'Борщ', 'Паста Карбонара']
   - 'как сделать панкейки?' (ожидался: Панкейки)
     Найдено: ['Борщ', 'Греческий салат', 'Шоколадный торт']
   - 'салат оливье рецепт' (ожидался: Оливье)
     Найдено: ['Паста Карбонара', 'Борщ', 'Греческий салат']
   - 'сырный суп как сварить' (ожидался: Сырный суп)
     Найдено: ['Борщ', 'Паста Карбонара', 'Греческий салат']
   - 'грибной крем-суп' (ожидался: Грибной крем-суп)
     Найдено: ['Греческий салат', 'Борщ', 'Паста Карбонара']

🔍 Тестирование пограничных запросов:

Запрос: 'суп с грибами'
Найденные рецепты: ['Греческий салат', 'Паста Карбонара', 'Борщ']

Запрос: 'завтрак сладкий'
Найденные рецепты: ['Борщ', 'Шоколадный торт', 'Паста Карбонара']

Запрос: 'мясное блюдо'
Найденные рецепты: ['Шоколадный торт', 'Борщ', 'Греч